In [252]:
import os 
import sys
import warnings
import subprocess
import pandas as pd
import numpy as np
import pybedtools
from Bio.Seq import Seq
from Bio.SeqUtils import gc_fraction
from gene2probe import *

This tutorial guides you through the design of custom probes against a gene of interest.

First, we need to specify the gene symbol, and which feature we are interested in designing probes agains (in this case we are using exons).
We also need to specify an output directory for the analysis.

### 1. Specify parameters

In [253]:
## Specify gene of interest and feature of interest
gene_ID = 'MIR4435-2HG'
mode = 'transcript' ## Whether to consider only exons / introns or full gene

## specify output directory
out_dir = '../sample_run/probeDesign_' + gene_ID + '_' + mode + '/'
## Create output directory
os.makedirs(out_dir, exist_ok=True)

Additionally, we need to provide the path to several resource files. Many of these files can be obtained from [UCSC table browser](https://genome.ucsc.edu/cgi-bin/hgTables).

We also need a blast database, such as the one we generated in the [previous tutorial](https://github.com/Teichlab/gene2probe/blob/main/notebooks/001_make_blast_database.ipynb).

In [254]:
## Required resources (most can be downloaded from 
gtf = '../hg38_resources/hg38.ncbiRefSeq.gtf' ## Gene annotation in gtf file
## We recommend using RefSeq as this is manually curated and more likely to contain an isoform that is present across most cell types
## Alternatively, one can filter based on RNA-seq data for a cell type/tissue of interest
fasta = '../hg38_resources/hg38.fa' ## Genome in fasta file
snp_db = '../hg38_resources/hg38_snp151Common.bed' ## Database of known SNPs and small indels
repeats = '../hg38_resources/hg38_rmsk.bed' ## bed file with repeats/low complexity regions to be excluded
gaps = '../hg38_resources/hg38_rmsk.bed' ## bed file with gaps in the genome assembly to be excluded
blast_db = '../hg38_resources/001_blastdb/hg38_ncbiRefSeq_transcripts_db' ## Database of all human transcripts to blast against

In [255]:
## Path to blast binaries.
## Replace with your conda environment
## This can also be omitted if you started the jupyter session from within the gene2probe conda environment
print('current working directory:', os.getcwd())

blast_exec_path = f"{os.environ['HOME']}/.miniforge3/envs/gene2probe_env/bin/"

if not os.path.isdir(blast_exec_path):
    warnings.warn(
        f'BLAST executable directory not found: {blast_exec_path}',
        RuntimeWarning,
    )

print('blast executable path:', blast_exec_path)

current working directory: /Users/dangriffiths/gene2probe/notebooks
blast executable path: /Users/dangriffiths/.miniforge3/envs/gene2probe_env/bin/


Finally, we need to provide a set of parameters related to our probe's length, at which nucleotide it's split (if at all), the acceptable range for GC content and any specific requirements for individual nucleotides.

Here we are following the [recommendations of 10x Genomics for custom probes for VisiumHD/VisiumFFPE/Flex](https://cdn.10xgenomics.com/image/upload/v1697739385/support-documents/CG000621_CustomProbeDesign_TechNote_RevC.pdf).

In [ ]:
## Additional parameters regarding how the probe should look like
probe_length = 50 ## Length of probe in nucleotides
split_nt = 25 ## Index of nucleotide to split the probe at (start of RHS) - set to None if splitting probe is not needed
min_GC = 0.44 ## Minimum GC content for probe (if split probe, applied to both LHS and RHS)
max_GC = 0.72 ## Maximum GC content for probe (if split probe, applied to both LHS and RHS)
required_nts = {24: 'T'} ## Dictionary of index (0-based) for required nts - by default, 25th nucleotide must be a T - set to None if no requirements
probe_offset = 1000 ## Minimum distance between probes - 10 bp is the recommended minimum by 10x, this can also be adjusted depending on how many probes pass other cutoffs
n_desired_probes =3 ## Number of probes to be designed.
min_mismatches = 5 ## Minimum number of mismatches (in at least LHS or RHS) - here we require in both to be more conservative

In [257]:
## Optionally, we can also specify adapters that have to be added to the probes.
## For example, for visiumHD:
LHS_pref = 'CCTTGGCACCCGAGAATTCCA' ## Will be added to the 5' of the LHS probe
LHS_suff = '' ## Will be added to the 3' of the LHS probe
RHS_pref = '/5Phos/' ## Will be added to the 5' of the RHS probe
RHS_suff = 'CCCATATAAGAAA' ## Will be added to the 3' of the RHS probe

## Leave as empty strings if you don't want to use them


### 2. Generate k-mers

Now we can start by reading the gene annotation and filtering for our gene of interest.

In [258]:
## Read gtf file
gene_anno = read_gtf(gtf)

In [259]:
gene_anno

,seqname,source,feature,start,end,score,strand,frame,attribute
0,chrM,ncbiRefSeq.2022-10-28,transcript,15956,16023,.,-,.,"gene_id ""TRNP""; transcript_id ""rna-TRNP""; gen..."
1,chrM,ncbiRefSeq.2022-10-28,exon,15956,16023,.,-,.,"gene_id ""TRNP""; transcript_id ""rna-TRNP""; exon..."
2,chrM,ncbiRefSeq.2022-10-28,transcript,15888,15953,.,+,.,"gene_id ""TRNT""; transcript_id ""rna-TRNT""; gen..."
3,chrM,ncbiRefSeq.2022-10-28,exon,15888,15953,.,+,.,"gene_id ""TRNT""; transcript_id ""rna-TRNT""; exon..."
4,chrM,ncbiRefSeq.2022-10-28,transcript,14747,15887,.,+,.,"gene_id ""CYTB""; transcript_id ""rna-CYTB""; gen..."
...,...,...,...,...,...,...,...,...,...
4886697,chr1,ncbiRefSeq.2022-10-28,exon,29321,29370,.,-,.,"gene_id ""WASH7P""; transcript_id ""NR_024540.1"";..."
4886698,chr1,ncbiRefSeq.2022-10-28,transcript,11874,14409,.,+,.,"gene_id ""DDX11L1""; transcript_id ""NR_046018.2""..."
4886699,chr1,ncbiRefSeq.2022-10-28,exon,11874,12227,.,+,.,"gene_id ""DDX11L1""; transcript_id ""NR_046018.2""..."
4886700,chr1,ncbiRefSeq.2022-10-28,exon,12613,12721,.,+,.,"gene_id ""DDX11L1""; transcript_id ""NR_046018.2""..."


In [260]:
## Extract regions corresponding to gene of interest (symbol: gene_name, Ensembl ID: gene_ID), subset to feature of interest and convert to bed style dataframe:
roi_bed = get_region_of_interest(gene_anno, gene_ID, gene_id_type = 'gene_name', feature=mode)

In [261]:
roi_bed

,seqname,start,end,name,score,strand
0,chr2,111429308,111495161,MIR4435-2HG_0,.,-
1,chr2,111207772,111495161,MIR4435-2HG_1,.,-
2,chr2,111207772,111495161,MIR4435-2HG_2,.,-
3,chr2,111207772,111495161,MIR4435-2HG_3,.,-
4,chr2,111207772,111495161,MIR4435-2HG_4,.,-
5,chr2,111195865,111495161,MIR4435-2HG_5,.,-
6,chr2,111195865,111495161,MIR4435-2HG_6,.,-
7,chr2,111195865,111495161,MIR4435-2HG_7,.,-


After extracting the coordinates of interest and converting to a bed-like format, we can generate all possible kmers that fall within these regions.

In [262]:
kmers = generate_kmers(roi_bed, k=probe_length)

In [263]:
kmers

,seqname,start,end,name,score,strand
0,chr2,111429308,111429358,MIR4435-2HG_0_0,.,-
1,chr2,111429309,111429359,MIR4435-2HG_0_1,.,-
2,chr2,111429310,111429360,MIR4435-2HG_0_2,.,-
3,chr2,111429311,111429361,MIR4435-2HG_0_3,.,-
4,chr2,111429312,111429362,MIR4435-2HG_0_4,.,-
...,...,...,...,...,...,...
2112900,chr2,111495107,111495157,MIR4435-2HG_7_299242,.,-
2112901,chr2,111495108,111495158,MIR4435-2HG_7_299243,.,-
2112902,chr2,111495109,111495159,MIR4435-2HG_7_299244,.,-
2112903,chr2,111495110,111495160,MIR4435-2HG_7_299245,.,-


In [264]:
## Export unfiltered
kmers.to_csv((out_dir + 'kmers_all.csv'))

### 3. Exclude annotated repeats/polymorphism

We can next exclude kmers overlapping undesired regions (repeats, low complexity regions, common polymorphism, gaps in the assembly) from further consideration.

Ideally, we will exclude everything that overlaps a repeat or polymorphism, but if we only have too few kmers available, we might need to relax these requirements (e.g., to only exclude kmers overlapping SNPs around the ligation junction, if the probes are split).

In [265]:
## For example, we could have removed all kmers overlapping a repeat/low complexity region within 5 nts of the ligation junction:
remove_overlaps(kmers, repeats, core=[20,30])

,seqname,start,end,name,score,strand
0,chr2,111429308,111429358,MIR4435-2HG_0_0,.,-
1,chr2,111429309,111429359,MIR4435-2HG_0_1,.,-
2,chr2,111429310,111429360,MIR4435-2HG_0_2,.,-
3,chr2,111429311,111429361,MIR4435-2HG_0_3,.,-
4,chr2,111429312,111429362,MIR4435-2HG_0_4,.,-
...,...,...,...,...,...,...
1239845,chr2,111495107,111495157,MIR4435-2HG_7_299242,.,-
1239846,chr2,111495108,111495158,MIR4435-2HG_7_299243,.,-
1239847,chr2,111495109,111495159,MIR4435-2HG_7_299244,.,-
1239848,chr2,111495110,111495160,MIR4435-2HG_7_299245,.,-


In [266]:
## In this case we have a lot of possible kmers, so we will remove those with overlaps in any part of the probe:
kmers = remove_overlaps(kmers, repeats)

In [267]:
kmers 

,seqname,start,end,name,score,strand
0,chr2,111429308,111429358,MIR4435-2HG_0_0,.,-
1,chr2,111429309,111429359,MIR4435-2HG_0_1,.,-
2,chr2,111429310,111429360,MIR4435-2HG_0_2,.,-
3,chr2,111429311,111429361,MIR4435-2HG_0_3,.,-
4,chr2,111429312,111429362,MIR4435-2HG_0_4,.,-
...,...,...,...,...,...,...
1198863,chr2,111495107,111495157,MIR4435-2HG_7_299242,.,-
1198864,chr2,111495108,111495158,MIR4435-2HG_7_299243,.,-
1198865,chr2,111495109,111495159,MIR4435-2HG_7_299244,.,-
1198866,chr2,111495110,111495160,MIR4435-2HG_7_299245,.,-


In [268]:
## Doing the same for gaps in the assembly (very unlikely since we are starting with annotated exons)
kmers = remove_overlaps(kmers, gaps)

In [ ]:
## And more importantly, against common polymorphism (SNPs, short indels)
kmers = remove_overlaps(kmers, snp_db)

In [ ]:
kmers

,seqname,start,end,name,score,strand
0,chr11_ML143358v1_fix,187697,187747,H19_0_0,.,-
1,chr11_ML143358v1_fix,187698,187748,H19_0_1,.,-
2,chr11_ML143358v1_fix,187699,187749,H19_0_2,.,-
3,chr11_ML143358v1_fix,187700,187750,H19_0_3,.,-
4,chr11_ML143358v1_fix,187701,187751,H19_0_4,.,-
...,...,...,...,...,...,...
19412,chr11,2001412,2001462,H19_5_6237,.,-
19413,chr11,2001413,2001463,H19_5_6238,.,-
19414,chr11,2001414,2001464,H19_5_6239,.,-
19415,chr11,2001415,2001465,H19_5_6240,.,-


### 4. Filter for desirable sequence features

Having excluded undesirable kmers based on intersection with genomic annotations, the next step is to consider their sequence features.

For this, we first extract the sequences of each k-mer, and then estimate features such as GC content and the presence of desired nucleotides in specific positions.

In [ ]:
## Get DNA for the transcript
kmers_bed = pybedtools.BedTool.from_dataframe(kmers)
kmers_seq = kmers_bed.sequence(fi=fasta, s=True) 

## We can read in the sequences and simultaneously monitor GC content and count the longest homopolymer stretch
kmers_seq_stats = get_sequence_stats(kmers_seq.seqfn, probe_length, split_nt)

WARNING. chromosome (chr11_ML143358v1_fix) was not found in the FASTA file. Skipping.
WARNING. chromosome (chr11_ML143358v1_fix) was not found in the FASTA file. Skipping.
WARNING. chromosome (chr11_ML143358v1_fix) was not found in the FASTA file. Skipping.
WARNING. chromosome (chr11_ML143358v1_fix) was not found in the FASTA file. Skipping.
WARNING. chromosome (chr11_ML143358v1_fix) was not found in the FASTA file. Skipping.
WARNING. chromosome (chr11_ML143358v1_fix) was not found in the FASTA file. Skipping.
WARNING. chromosome (chr11_ML143358v1_fix) was not found in the FASTA file. Skipping.
WARNING. chromosome (chr11_ML143358v1_fix) was not found in the FASTA file. Skipping.
WARNING. chromosome (chr11_ML143358v1_fix) was not found in the FASTA file. Skipping.
WARNING. chromosome (chr11_ML143358v1_fix) was not found in the FASTA file. Skipping.
WARNING. chromosome (chr11_ML143358v1_fix) was not found in the FASTA file. Skipping.
WARNING. chromosome (chr11_ML143358v1_fix) was not fou

In [ ]:
## Combining with our dataframe
kmers = pd.merge(kmers, kmers_seq_stats, left_index=True, right_index=True)

In [ ]:
kmers

,seqname,start,end,name,score,strand,kmer_coord,transcript_seq,probe_seq,GC_content_full,longest_homopolymer,GC_content_LHS,GC_content_RHS
0,chr11_ML143358v1_fix,187697,187747,H19_0_0,.,-,chr11:1995175-1995225(-),GGCTTCAGCAGGAGCCCTGGACTCATCATCAATAAACACTGTTACA...,TTGCTGTAACAGTGTTTATTGATGATGAGTCCAGGGCTCCTGCTGA...,0.48,3,0.32,0.64
1,chr11_ML143358v1_fix,187698,187748,H19_0_1,.,-,chr11:1995176-1995226(-),GGGCTTCAGCAGGAGCCCTGGACTCATCATCAATAAACACTGTTAC...,TGCTGTAACAGTGTTTATTGATGATGAGTCCAGGGCTCCTGCTGAA...,0.50,3,0.32,0.68
2,chr11_ML143358v1_fix,187699,187749,H19_0_2,.,-,chr11:1995177-1995227(-),AGGGCTTCAGCAGGAGCCCTGGACTCATCATCAATAAACACTGTTA...,GCTGTAACAGTGTTTATTGATGATGAGTCCAGGGCTCCTGCTGAAG...,0.50,3,0.36,0.64
3,chr11_ML143358v1_fix,187700,187750,H19_0_3,.,-,chr11:1995178-1995228(-),CAGGGCTTCAGCAGGAGCCCTGGACTCATCATCAATAAACACTGTT...,CTGTAACAGTGTTTATTGATGATGAGTCCAGGGCTCCTGCTGAAGC...,0.50,3,0.32,0.68
4,chr11_ML143358v1_fix,187701,187751,H19_0_4,.,-,chr11:1995179-1995229(-),CCAGGGCTTCAGCAGGAGCCCTGGACTCATCATCAATAAACACTGT...,TGTAACAGTGTTTATTGATGATGAGTCCAGGGCTCCTGCTGAAGCC...,0.50,3,0.32,0.68
...,...,...,...,...,...,...,...,...,...,...,...,...,...
8114,chr11_ML143358v1_fix,190755,190805,H19_2_3058,.,-,chr11:2001412-2001462(-),GACCATGGCCCCGTATCACCTGGGTCAGGCACTGAAGCTGGGACAG...,TCTCCTGTCCCAGCTTCAGTGCCTGACCCAGGTGATACGGGGCCAT...,0.62,4,0.60,0.64
8115,chr11_ML143358v1_fix,190756,190806,H19_2_3059,.,-,chr11:2001413-2001463(-),GGACCATGGCCCCGTATCACCTGGGTCAGGCACTGAAGCTGGGACA...,CTCCTGTCCCAGCTTCAGTGCCTGACCCAGGTGATACGGGGCCATG...,0.64,4,0.60,0.68
8116,chr11_ML143358v1_fix,190757,190807,H19_2_3060,.,-,chr11:2001414-2001464(-),AGGACCATGGCCCCGTATCACCTGGGTCAGGCACTGAAGCTGGGAC...,TCCTGTCCCAGCTTCAGTGCCTGACCCAGGTGATACGGGGCCATGG...,0.62,4,0.60,0.64
8117,chr11_ML143358v1_fix,190758,190808,H19_2_3061,.,-,chr11:2001415-2001465(-),GAGGACCATGGCCCCGTATCACCTGGGTCAGGCACTGAAGCTGGGA...,CCTGTCCCAGCTTCAGTGCCTGACCCAGGTGATACGGGGCCATGGT...,0.64,4,0.64,0.64


In [ ]:
## Check for required nucleotides in specific positions:
if required_nts is not None:
    kmers['has_required_nts'] = check_for_required_nts(kmers, required_nts)
    print(kmers['has_required_nts'].value_counts())
    ## Filter for required nucleotides
    kmers = kmers[kmers['has_required_nts']==True].reset_index(drop=True)

has_required_nts
False    6576
True     1543
Name: count, dtype: int64


In [ ]:
## Export kmers before filtering
kmers.to_csv((out_dir + 'kmers_candidates_unfiltered.csv'))

In [ ]:
kmers

,seqname,start,end,name,score,strand,kmer_coord,transcript_seq,probe_seq,GC_content_full,longest_homopolymer,GC_content_LHS,GC_content_RHS,has_required_nts
0,chr11_ML143358v1_fix,187698,187748,H19_0_1,.,-,chr11:1995176-1995226(-),GGGCTTCAGCAGGAGCCCTGGACTCATCATCAATAAACACTGTTAC...,TGCTGTAACAGTGTTTATTGATGATGAGTCCAGGGCTCCTGCTGAA...,0.50,3,0.32,0.68,True
1,chr11_ML143358v1_fix,187702,187752,H19_0_5,.,-,chr11:1995180-1995230(-),ACCAGGGCTTCAGCAGGAGCCCTGGACTCATCATCAATAAACACTG...,GTAACAGTGTTTATTGATGATGAGTCCAGGGCTCCTGCTGAAGCCC...,0.50,3,0.32,0.68,True
2,chr11_ML143358v1_fix,187710,187760,H19_0_13,.,-,chr11:1995188-1995238(-),CCCTCCCCACCAGGGCTTCAGCAGGAGCCCTGGACTCATCATCAAT...,GTTTATTGATGATGAGTCCAGGGCTCCTGCTGAAGCCCTGGTGGGG...,0.58,4,0.44,0.72,True
3,chr11_ML143358v1_fix,187713,187763,H19_0_16,.,-,chr11:1995191-1995241(-),TGCCCCTCCCCACCAGGGCTTCAGCAGGAGCCCTGGACTCATCATC...,TATTGATGATGAGTCCAGGGCTCCTGCTGAAGCCCTGGTGGGGAGG...,0.60,4,0.48,0.72,True
4,chr11_ML143358v1_fix,187716,187766,H19_0_19,.,-,chr11:1995194-1995244(-),CTGTGCCCCTCCCCACCAGGGCTTCAGCAGGAGCCCTGGACTCATC...,TGATGATGAGTCCAGGGCTCCTGCTGAAGCCCTGGTGGGGAGGGGC...,0.64,4,0.56,0.72,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1538,chr11_ML143358v1_fix,190738,190788,H19_2_3041,.,-,chr11:2001395-2001445(-),ACCTGGGTCAGGCACTGAAGCTGGGACAGGAGAGCAGAGACTTCCA...,ATTTTGGAAGTCTCTGCTCTCCTGTCCCAGCTTCAGTGCCTGACCC...,0.54,4,0.44,0.64,True
1539,chr11_ML143358v1_fix,190745,190795,H19_2_3048,.,-,chr11:2001402-2001452(-),CCGTATCACCTGGGTCAGGCACTGAAGCTGGGACAGGAGAGCAGAG...,AAGTCTCTGCTCTCCTGTCCCAGCTTCAGTGCCTGACCCAGGTGAT...,0.58,3,0.56,0.60,True
1540,chr11_ML143358v1_fix,190746,190796,H19_2_3049,.,-,chr11:2001403-2001453(-),CCCGTATCACCTGGGTCAGGCACTGAAGCTGGGACAGGAGAGCAGA...,AGTCTCTGCTCTCCTGTCCCAGCTTCAGTGCCTGACCCAGGTGATA...,0.60,3,0.56,0.64,True
1541,chr11_ML143358v1_fix,190750,190800,H19_2_3053,.,-,chr11:2001407-2001457(-),TGGCCCCGTATCACCTGGGTCAGGCACTGAAGCTGGGACAGGAGAG...,TCTGCTCTCCTGTCCCAGCTTCAGTGCCTGACCCAGGTGATACGGG...,0.62,4,0.56,0.68,True


In [ ]:
## Filter for GC content
kmers = filter_by_GC_content(kmers, min_GC, max_GC)

In [ ]:
kmers

,seqname,start,end,name,score,strand,kmer_coord,transcript_seq,probe_seq,GC_content_full,longest_homopolymer,GC_content_LHS,GC_content_RHS,has_required_nts
0,chr11_ML143358v1_fix,187724,187774,H19_0_27,.,-,chr11:1995202-1995252(-),ATCTCGCTCTGTGCCCCTCCCCACCAGGGCTTCAGCAGGAGCCCTG...,AGTCCAGGGCTCCTGCTGAAGCCCTGGTGGGGAGGGGCACAGAGCG...,0.66,4,0.64,0.68,True
1,chr11_ML143358v1_fix,187749,187799,H19_0_52,.,-,chr11:1995227-1995277(-),GCCTTCAAGCATTCCATTACGCCCCATCTCGCTCTGTGCCCCTCCC...,GGTGGGGAGGGGCACAGAGCGAGATGGGGCGTAATGGAATGCTTGA...,0.62,4,0.68,0.56,True
2,chr11_ML143358v1_fix,187756,187806,H19_0_59,.,-,chr11:1995234-1995284(-),CGGAGCAGCCTTCAAGCATTCCATTACGCCCCATCTCGCTCTGTGC...,AGGGGCACAGAGCGAGATGGGGCGTAATGGAATGCTTGAAGGCTGC...,0.60,4,0.68,0.52,True
3,chr11_ML143358v1_fix,187759,187809,H19_0_62,.,-,chr11:1995237-1995287(-),TCACGGAGCAGCCTTCAAGCATTCCATTACGCCCCATCTCGCTCTG...,GGCACAGAGCGAGATGGGGCGTAATGGAATGCTTGAAGGCTGCTCC...,0.58,4,0.60,0.56,True
4,chr11_ML143358v1_fix,187764,187814,H19_0_67,.,-,chr11:1995242-1995292(-),CGACATCACGGAGCAGCCTTCAAGCATTCCATTACGCCCCATCTCG...,AGAGCGAGATGGGGCGTAATGGAATGCTTGAAGGCTGCTCCGTGAT...,0.56,4,0.52,0.60,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
782,chr11_ML143358v1_fix,190733,190783,H19_2_3036,.,-,chr11:2001390-2001440(-),GGTCAGGCACTGAAGCTGGGACAGGAGAGCAGAGACTTCCAAAATG...,CCCTCATTTTGGAAGTCTCTGCTCTCCTGTCCCAGCTTCAGTGCCT...,0.56,4,0.48,0.64,True
783,chr11_ML143358v1_fix,190745,190795,H19_2_3048,.,-,chr11:2001402-2001452(-),CCGTATCACCTGGGTCAGGCACTGAAGCTGGGACAGGAGAGCAGAG...,AAGTCTCTGCTCTCCTGTCCCAGCTTCAGTGCCTGACCCAGGTGAT...,0.58,3,0.56,0.60,True
784,chr11_ML143358v1_fix,190746,190796,H19_2_3049,.,-,chr11:2001403-2001453(-),CCCGTATCACCTGGGTCAGGCACTGAAGCTGGGACAGGAGAGCAGA...,AGTCTCTGCTCTCCTGTCCCAGCTTCAGTGCCTGACCCAGGTGATA...,0.60,3,0.56,0.64,True
785,chr11_ML143358v1_fix,190750,190800,H19_2_3053,.,-,chr11:2001407-2001457(-),TGGCCCCGTATCACCTGGGTCAGGCACTGAAGCTGGGACAGGAGAG...,TCTGCTCTCCTGTCCCAGCTTCAGTGCCTGACCCAGGTGATACGGG...,0.62,4,0.56,0.68,True


In [ ]:
## Candidate kmers
kmers

,seqname,start,end,name,score,strand,kmer_coord,transcript_seq,probe_seq,GC_content_full,longest_homopolymer,GC_content_LHS,GC_content_RHS,has_required_nts
0,chr11_ML143358v1_fix,187724,187774,H19_0_27,.,-,chr11:1995202-1995252(-),ATCTCGCTCTGTGCCCCTCCCCACCAGGGCTTCAGCAGGAGCCCTG...,AGTCCAGGGCTCCTGCTGAAGCCCTGGTGGGGAGGGGCACAGAGCG...,0.66,4,0.64,0.68,True
1,chr11_ML143358v1_fix,187749,187799,H19_0_52,.,-,chr11:1995227-1995277(-),GCCTTCAAGCATTCCATTACGCCCCATCTCGCTCTGTGCCCCTCCC...,GGTGGGGAGGGGCACAGAGCGAGATGGGGCGTAATGGAATGCTTGA...,0.62,4,0.68,0.56,True
2,chr11_ML143358v1_fix,187756,187806,H19_0_59,.,-,chr11:1995234-1995284(-),CGGAGCAGCCTTCAAGCATTCCATTACGCCCCATCTCGCTCTGTGC...,AGGGGCACAGAGCGAGATGGGGCGTAATGGAATGCTTGAAGGCTGC...,0.60,4,0.68,0.52,True
3,chr11_ML143358v1_fix,187759,187809,H19_0_62,.,-,chr11:1995237-1995287(-),TCACGGAGCAGCCTTCAAGCATTCCATTACGCCCCATCTCGCTCTG...,GGCACAGAGCGAGATGGGGCGTAATGGAATGCTTGAAGGCTGCTCC...,0.58,4,0.60,0.56,True
4,chr11_ML143358v1_fix,187764,187814,H19_0_67,.,-,chr11:1995242-1995292(-),CGACATCACGGAGCAGCCTTCAAGCATTCCATTACGCCCCATCTCG...,AGAGCGAGATGGGGCGTAATGGAATGCTTGAAGGCTGCTCCGTGAT...,0.56,4,0.52,0.60,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
782,chr11_ML143358v1_fix,190733,190783,H19_2_3036,.,-,chr11:2001390-2001440(-),GGTCAGGCACTGAAGCTGGGACAGGAGAGCAGAGACTTCCAAAATG...,CCCTCATTTTGGAAGTCTCTGCTCTCCTGTCCCAGCTTCAGTGCCT...,0.56,4,0.48,0.64,True
783,chr11_ML143358v1_fix,190745,190795,H19_2_3048,.,-,chr11:2001402-2001452(-),CCGTATCACCTGGGTCAGGCACTGAAGCTGGGACAGGAGAGCAGAG...,AAGTCTCTGCTCTCCTGTCCCAGCTTCAGTGCCTGACCCAGGTGAT...,0.58,3,0.56,0.60,True
784,chr11_ML143358v1_fix,190746,190796,H19_2_3049,.,-,chr11:2001403-2001453(-),CCCGTATCACCTGGGTCAGGCACTGAAGCTGGGACAGGAGAGCAGA...,AGTCTCTGCTCTCCTGTCCCAGCTTCAGTGCCTGACCCAGGTGATA...,0.60,3,0.56,0.64,True
785,chr11_ML143358v1_fix,190750,190800,H19_2_3053,.,-,chr11:2001407-2001457(-),TGGCCCCGTATCACCTGGGTCAGGCACTGAAGCTGGGACAGGAGAG...,TCTGCTCTCCTGTCCCAGCTTCAGTGCCTGACCCAGGTGATACGGG...,0.62,4,0.56,0.68,True


In [ ]:
kmers.to_csv((out_dir + 'kmers_candidates_filtered.csv'))

### 5. Remove probes with potential off-targets

Having identified a set of kmers that fulfill our sequence requirements, we can next proceed with testing whether they are specific to our transcript/exon of interest.

For this we rely on using BLAST. 

We recommend blasting against transcripts (i.e., exons and introns combined) to be as conservative as possible in terms of off-targets. 
However, in cases where it is not possible to obtain enough suitable kmers (e.g., for  short transcripts), it is reasonable to relax this requirement by BLASTing against exons only (a much smaller search space).

At this step, we also want to consider whether our probes are split (as in the current specifications for VisiumHD) or a single oligo. If probes are split, it's best to BLAST each side separately, to make sure that both sides are specific.

In [ ]:
## The first thing to do is to export our sequences in fasta format, so that we can use them for BLAST
write_fasta(kmers['name'], kmers['transcript_seq'], (out_dir + 'kmers_candidates_filtered_transcript_seqs.fa'))
## If our probes are meant to be split, we should additionally blast them separately 
## Note that the LHS/RHS in the transcript are reversed compared to the probe (i.e., the LHS of the transcript is complementary to the RHS of the probe)
if split_nt is not None: 
    ## Make split probes
    kmers['transcript_seq_LHS'] = [seq[0:split_nt] for seq in kmers['transcript_seq']]
    kmers['transcript_seq_RHS'] = [seq[split_nt: probe_length] for seq in kmers['transcript_seq']]

    ## We are exporting the transcript sequence as that's the one that has to be blasted against the human transcriptome
    write_fasta(kmers['name'], kmers['transcript_seq_LHS'], (out_dir + 'kmers_candidates_filtered_transcript_seqs_LHS.fa'))
    write_fasta(kmers['name'], kmers['transcript_seq_RHS'], (out_dir + 'kmers_candidates_filtered_transcript_seqs_RHS.fa'))    

In [ ]:
blast_res = {}
## First, blast the full probe
blast_res['full'] = run_blast(fasta=(out_dir + 'kmers_candidates_filtered_transcript_seqs.fa'),
                              blastdb = blast_db,
                              path2blastn=(blast_exec_path + 'blastn'),
                              outfile = (out_dir + 'kmers_candidates_filtered_blast_output.txt'))

## Additionally, if probe is split, blast each side separately
if split_nt is not None: 
    blast_res['LHS'] = run_blast(fasta=(out_dir + 'kmers_candidates_filtered_transcript_seqs_LHS.fa'),
                                     blastdb = blast_db,
                                     path2blastn=(blast_exec_path + 'blastn'),
                                     outfile = (out_dir + 'kmers_candidates_filtered_blast_output_LHS.txt'))
    blast_res['RHS'] = run_blast(fasta=(out_dir + 'kmers_candidates_filtered_transcript_seqs_RHS.fa'),
                                     blastdb = blast_db,
                                     path2blastn=(blast_exec_path + 'blastn'),
                                     outfile = (out_dir + 'kmers_candidates_filtered_blast_output_RHS.txt'))

In [ ]:
for k in blast_res.keys():
    print(("The following genes were detected in mode: " +  k))
    print(blast_res[k]['sgeneid'].value_counts().head(10))

The following genes were detected in mode: full
sgeneid
H19        1984
ZFYVE27     183
FTO         156
APBB2       153
SGPP2       144
TSC22D3     108
WSCD2        78
CREBRF       64
SLC4A3       48
DNAJB12      45
Name: count, dtype: int64
The following genes were detected in mode: LHS
sgeneid
H19        1907
LCA5L       342
ACTR3C      126
SGPP2       117
ZFAT        117
DLG2        114
BNC2         80
BCAT1        66
DPP6         60
RPS6KA2      54
Name: count, dtype: int64
The following genes were detected in mode: RHS
sgeneid
H19         1895
CABIN1       144
BNC2         120
PPP2R3B       96
NTM           66
VEPH1         48
SFTPC         45
FTO           39
TFB1M         36
SLC25A43      33
Name: count, dtype: int64


In [ ]:
blast_res['full']

,name,sseqid,pident,length,mismatch,gapopen,qstart,qend,sstart,send,evalue,bitscore,sgeneid
0,H19_0_27,H19::chr11:1995175-2001466(-),100.000,50,0,0,1,50,6215,6264,8.400000e-17,91.5,H19
1,H19_0_27,H19::chr11:1995175-1997875(-),100.000,50,0,0,1,50,2624,2673,8.400000e-17,91.5,H19
2,H19_0_27,H19::chr11:1995175-1997875(-),100.000,50,0,0,1,50,2624,2673,8.400000e-17,91.5,H19
3,H19_0_27,GUSBP6::chr7:64100304-64120545(+),90.323,31,3,0,19,49,13017,13047,3.800000e-02,43.7,GUSBP6
4,H19_0_27,TEAD4::chr12:2959396-3040676(+),86.111,36,4,1,13,47,61032,61067,4.700000e-01,40.1,TEAD4
...,...,...,...,...,...,...,...,...,...,...,...,...,...
3463,H19_2_3036,H19::chr11:1995175-2001466(-),100.000,50,0,0,1,50,27,76,8.400000e-17,91.5,H19
3464,H19_2_3048,H19::chr11:1995175-2001466(-),100.000,50,0,0,1,50,15,64,8.400000e-17,91.5,H19
3465,H19_2_3049,H19::chr11:1995175-2001466(-),100.000,50,0,0,1,50,14,63,8.400000e-17,91.5,H19
3466,H19_2_3053,H19::chr11:1995175-2001466(-),100.000,50,0,0,1,50,10,59,8.400000e-17,91.5,H19


Not all BLAST hits will be off-targets. Hopefully, our gene of interest is included in the BLAST output. We therefore need to filter for hits with different gene IDs.

In [ ]:
offtargets = []
for k in blast_res.keys():
    offtargets += (detect_offtargets(blast_res[k], gene_ID, min_mismatches=min_mismatches))
## Remove redundancies
offtargets = list(set(offtargets))

In [ ]:
len(offtargets)

313

In [ ]:
## Remove off-targets
kmers = kmers[kmers['name'].isin(offtargets)==False].reset_index(drop=True)

In [ ]:
kmers 

,seqname,start,end,name,score,strand,kmer_coord,transcript_seq,probe_seq,GC_content_full,longest_homopolymer,GC_content_LHS,GC_content_RHS,has_required_nts,transcript_seq_LHS,transcript_seq_RHS
0,chr11_ML143358v1_fix,187749,187799,H19_0_52,.,-,chr11:1995227-1995277(-),GCCTTCAAGCATTCCATTACGCCCCATCTCGCTCTGTGCCCCTCCC...,GGTGGGGAGGGGCACAGAGCGAGATGGGGCGTAATGGAATGCTTGA...,0.62,4,0.68,0.56,True,GCCTTCAAGCATTCCATTACGCCCC,ATCTCGCTCTGTGCCCCTCCCCACC
1,chr11_ML143358v1_fix,187756,187806,H19_0_59,.,-,chr11:1995234-1995284(-),CGGAGCAGCCTTCAAGCATTCCATTACGCCCCATCTCGCTCTGTGC...,AGGGGCACAGAGCGAGATGGGGCGTAATGGAATGCTTGAAGGCTGC...,0.60,4,0.68,0.52,True,CGGAGCAGCCTTCAAGCATTCCATT,ACGCCCCATCTCGCTCTGTGCCCCT
2,chr11_ML143358v1_fix,187759,187809,H19_0_62,.,-,chr11:1995237-1995287(-),TCACGGAGCAGCCTTCAAGCATTCCATTACGCCCCATCTCGCTCTG...,GGCACAGAGCGAGATGGGGCGTAATGGAATGCTTGAAGGCTGCTCC...,0.58,4,0.60,0.56,True,TCACGGAGCAGCCTTCAAGCATTCC,ATTACGCCCCATCTCGCTCTGTGCC
3,chr11_ML143358v1_fix,187764,187814,H19_0_67,.,-,chr11:1995242-1995292(-),CGACATCACGGAGCAGCCTTCAAGCATTCCATTACGCCCCATCTCG...,AGAGCGAGATGGGGCGTAATGGAATGCTTGAAGGCTGCTCCGTGAT...,0.56,4,0.52,0.60,True,CGACATCACGGAGCAGCCTTCAAGC,ATTCCATTACGCCCCATCTCGCTCT
4,chr11_ML143358v1_fix,187767,187817,H19_0_70,.,-,chr11:1995245-1995295(-),GACCGACATCACGGAGCAGCCTTCAAGCATTCCATTACGCCCCATC...,GCGAGATGGGGCGTAATGGAATGCTTGAAGGCTGCTCCGTGATGTC...,0.58,4,0.56,0.60,True,GACCGACATCACGGAGCAGCCTTCA,AGCATTCCATTACGCCCCATCTCGC
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
469,chr11_ML143358v1_fix,190712,190762,H19_2_3015,.,-,chr11:2001369-2001419(-),CAGGAGAGCAGAGACTTCCAAAATGAGGGATCCCTGTGTTCTGAGG...,ATCACCTCAGAACACAGGGATCCCTCATTTTGGAAGTCTCTGCTCT...,0.50,4,0.52,0.48,True,CAGGAGAGCAGAGACTTCCAAAATG,AGGGATCCCTGTGTTCTGAGGTGAT
470,chr11_ML143358v1_fix,190715,190765,H19_2_3018,.,-,chr11:2001372-2001422(-),GGACAGGAGAGCAGAGACTTCCAAAATGAGGGATCCCTGTGTTCTG...,ACCTCAGAACACAGGGATCCCTCATTTTGGAAGTCTCTGCTCTCCT...,0.52,4,0.52,0.52,True,GGACAGGAGAGCAGAGACTTCCAAA,ATGAGGGATCCCTGTGTTCTGAGGT
471,chr11_ML143358v1_fix,190716,190766,H19_2_3019,.,-,chr11:2001373-2001423(-),GGGACAGGAGAGCAGAGACTTCCAAAATGAGGGATCCCTGTGTTCT...,CCTCAGAACACAGGGATCCCTCATTTTGGAAGTCTCTGCTCTCCTG...,0.54,4,0.52,0.56,True,GGGACAGGAGAGCAGAGACTTCCAA,AATGAGGGATCCCTGTGTTCTGAGG
472,chr11_ML143358v1_fix,190717,190767,H19_2_3020,.,-,chr11:2001374-2001424(-),TGGGACAGGAGAGCAGAGACTTCCAAAATGAGGGATCCCTGTGTTC...,CTCAGAACACAGGGATCCCTCATTTTGGAAGTCTCTGCTCTCCTGT...,0.52,4,0.48,0.56,True,TGGGACAGGAGAGCAGAGACTTCCA,AAATGAGGGATCCCTGTGTTCTGAG


### 6. Select non-overlapping probes

At this point, we have effectively acquired a set of usable probes. They don't overlap undesirable regions (repeats/polymorphism), have desirable sequence features (GC content, specific nucleotides) and are specific to our gene of interest.

In this particular case, we still have a lot of possible k-mers (much more than the number of probes we intend to design). We can therefore choose to prioritise k-mers with shorter homopolymer stretches, as these are also discouraged by the [10x recommendations](https://cdn.10xgenomics.com/image/upload/v1697739385/support-documents/CG000621_CustomProbeDesign_TechNote_RevC.pdf).

However, at this stage you might want to consider ranking probes in a diffferent way, depending on your application.

After having ranked our k-mers in whatever way we think is reasonable at this stage, we can proceed with selecting the top probe, then removing all overlapping/adjacent probes (within a window determined by `probe_offset`).

Since we have so many available probes, we will be increasing `probe_offset` from `100 (default)` to `1000`.

In [ ]:
## Sort in increasing homopolymer length
kmers = kmers.sort_values('longest_homopolymer', ascending=True).reset_index(drop=True)

In [ ]:
kmers

,seqname,start,end,name,score,strand,kmer_coord,transcript_seq,probe_seq,GC_content_full,longest_homopolymer,GC_content_LHS,GC_content_RHS,has_required_nts,transcript_seq_LHS,transcript_seq_RHS
0,chr11_ML143358v1_fix,189769,189819,H19_1_2072,.,-,chr11:1996324-1996374(-),AGACGCCAGGTCCGGTGGACGTGACAAGCAGGACATGACATGGTCC...,CACCGGACCATGTCATGTCCTGCTTGTCACGTCCACCGGACCTGGC...,0.62,2,0.56,0.68,True,AGACGCCAGGTCCGGTGGACGTGAC,AAGCAGGACATGACATGGTCCGGTG
1,chr11_ML143358v1_fix,189764,189814,H19_0_2067,.,-,chr11:1995263-1995313(-),GCCTAGTCTGGAAGCTCCGACCGACATCACGGAGCAGCCTTCAAGC...,GAATGCTTGAAGGCTGCTCCGTGATGTCGGTCGGAGCTTCCAGACT...,0.58,2,0.52,0.64,True,GCCTAGTCTGGAAGCTCCGACCGAC,ATCACGGAGCAGCCTTCAAGCATTC
2,chr11_ML143358v1_fix,189761,189811,H19_0_2064,.,-,chr11:1995260-1995310(-),TAGTCTGGAAGCTCCGACCGACATCACGGAGCAGCCTTCAAGCATT...,ATGGAATGCTTGAAGGCTGCTCCGTGATGTCGGTCGGAGCTTCCAG...,0.54,2,0.52,0.56,True,TAGTCTGGAAGCTCCGACCGACATC,ACGGAGCAGCCTTCAAGCATTCCAT
3,chr11_ML143358v1_fix,189757,189807,H19_0_2060,.,-,chr11:1995256-1995306(-),CTGGAAGCTCCGACCGACATCACGGAGCAGCCTTCAAGCATTCCAT...,CGTAATGGAATGCTTGAAGGCTGCTCCGTGATGTCGGTCGGAGCTT...,0.56,2,0.48,0.64,True,CTGGAAGCTCCGACCGACATCACGG,AGCAGCCTTCAAGCATTCCATTACG
4,chr11_ML143358v1_fix,187790,187840,H19_1_93,.,-,chr11:1996324-1996374(-),AGACGCCAGGTCCGGTGGACGTGACAAGCAGGACATGACATGGTCC...,CACCGGACCATGTCATGTCCTGCTTGTCACGTCCACCGGACCTGGC...,0.62,2,0.56,0.68,True,AGACGCCAGGTCCGGTGGACGTGAC,AAGCAGGACATGACATGGTCCGGTG
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
469,chr11_ML143358v1_fix,187708,187758,H19_2_11,.,-,chr11:1997234-1997284(-),GGGCAGGAGAGTTAGCAAAGGTGACATCTTCTCGGGGGGAGCCGAG...,CAGTCTCGGCTCCCCCCGAGAAGATGTCACCTTTGCTAACTCTCCT...,0.60,6,0.64,0.56,True,GGGCAGGAGAGTTAGCAAAGGTGAC,ATCTTCTCGGGGGGAGCCGAGACTG
470,chr11_ML143358v1_fix,188380,188430,H19_1_683,.,-,chr11:1997234-1997284(-),GGGCAGGAGAGTTAGCAAAGGTGACATCTTCTCGGGGGGAGCCGAG...,CAGTCTCGGCTCCCCCCGAGAAGATGTCACCTTTGCTAACTCTCCT...,0.60,6,0.64,0.56,True,GGGCAGGAGAGTTAGCAAAGGTGAC,ATCTTCTCGGGGGGAGCCGAGACTG
471,chr11_ML143358v1_fix,188387,188437,H19_1_690,.,-,chr11:1997241-1997291(-),AGGATGGGGGCAGGAGAGTTAGCAAAGGTGACATCTTCTCGGGGGG...,GGCTCCCCCCGAGAAGATGTCACCTTTGCTAACTCTCCTGCCCCCA...,0.60,6,0.64,0.56,True,AGGATGGGGGCAGGAGAGTTAGCAA,AGGTGACATCTTCTCGGGGGGAGCC
472,chr11_ML143358v1_fix,189052,189102,H19_0_1355,.,-,chr11:1997234-1997284(-),GGGCAGGAGAGTTAGCAAAGGTGACATCTTCTCGGGGGGAGCCGAG...,CAGTCTCGGCTCCCCCCGAGAAGATGTCACCTTTGCTAACTCTCCT...,0.60,6,0.64,0.56,True,GGGCAGGAGAGTTAGCAAAGGTGAC,ATCTTCTCGGGGGGAGCCGAGACTG


In [ ]:
## We have a lot of probes here - increasing the offset to 1000 bp to space them out
# probe_offset = 1000 # uncomment to make changes to the top-level arguments 

In [ ]:
# Select probes by row coordinates instead of a potentially duplicated name.
selected_probes_list = []

while len(selected_probes_list) < n_desired_probes and not df.empty:
    selected_probe = df.iloc[[0]].copy()
    selected_probes_list.append(selected_probe)

    start = int(selected_probe["start"].iloc[0])
    end = int(selected_probe["end"].iloc[0])

    df = (
        df.loc[
            (df["end"] < start - probe_offset)
            | (df["start"] > end + probe_offset)
        ]
        .reset_index(drop=True)
        .copy()
    )

selected_probes = pd.concat(selected_probes_list, ignore_index=True)

In [ ]:
selected_probes_df = pd.concat(selected_probes_list, axis=0).reset_index(drop=True)

In [ ]:
selected_probes_df

,seqname,start,end,name,score,strand,kmer_coord,transcript_seq,probe_seq,GC_content_full,longest_homopolymer,GC_content_LHS,GC_content_RHS,has_required_nts,transcript_seq_LHS,transcript_seq_RHS
0,chr9,22058363,22058413,CDKN2B-AS1_8_63573,.,+,chr9:22058363-22058413(+),CCAATGAACGCCTTCACTGATATCCAAAGCATGAAGGACACACCAG...,TTCCCTGGTGTGTCCTTCATGCTTTGGATATCAGTGAAGGCGTTCA...,0.48,3,0.48,0.48,True,CCAATGAACGCCTTCACTGATATCC,AAAGCATGAAGGACACACCAGGGAA
1,chr9,22050400,22050450,CDKN2B-AS1_8_55610,.,+,chr9:22050400-22050450(+),GAAGTACTTCTTTGGCTGCCAAGGAAGTCCCAGCTGAAAGGTAACC...,CTTTGGTTACCTTTCAGCTGGGACTTCCTTGGCAGCCAAAGAAGTA...,0.48,3,0.48,0.48,True,GAAGTACTTCTTTGGCTGCCAAGGA,AGTCCCAGCTGAAAGGTAACCAAAG
2,chr9,22050830,22050880,CDKN2B-AS1_8_56040,.,+,chr9:22050830-22050880(+),CCTTCAAATAGTTCACCTCAGTGGGATCCAGTGTGTGACCTGCAAA...,GCACTTTGCAGGTCACACACTGGATCCCACTGAGGTGAACTATTTG...,0.50,3,0.52,0.48,True,CCTTCAAATAGTTCACCTCAGTGGG,ATCCAGTGTGTGACCTGCAAAGTGC


In [ ]:
for seq in selected_probes_df['transcript_seq']:
    print(seq)

CCAATGAACGCCTTCACTGATATCCAAAGCATGAAGGACACACCAGGGAA
GAAGTACTTCTTTGGCTGCCAAGGAAGTCCCAGCTGAAAGGTAACCAAAG
CCTTCAAATAGTTCACCTCAGTGGGATCCAGTGTGTGACCTGCAAAGTGC


And we are done! We have now selected three potential probes for our gene.

If our probes are meant to be split, we can additionally generate these columns:

In [ ]:
if split_nt is not None: 
    ## Make split probes (and add adapters if provided)
    selected_probes_df['probe_seq_LHS'] = [(LHS_pref + seq[0:split_nt] + LHS_suff) for seq in selected_probes_df['probe_seq']]
    selected_probes_df['probe_seq_RHS'] = [(RHS_pref +seq[split_nt: probe_length] + RHS_suff) for seq in selected_probes_df['probe_seq']]

In [ ]:
## Also add gene_ID for completeness
selected_probes_df['gene_ID'] = gene_ID

In [ ]:
## Export selected probes as dataframe:
selected_probes_df.to_csv((out_dir + 'kmers_selected_probes.csv'))

We always recommend additionally performing a manual BLAST of these probe sequences to make sure that there are no off-target effects.

In [ ]:
selected_probes_df

,seqname,start,end,name,score,strand,kmer_coord,transcript_seq,probe_seq,GC_content_full,longest_homopolymer,GC_content_LHS,GC_content_RHS,has_required_nts,transcript_seq_LHS,transcript_seq_RHS,probe_seq_LHS,probe_seq_RHS,gene_ID
0,chr9,22058363,22058413,CDKN2B-AS1_8_63573,.,+,chr9:22058363-22058413(+),CCAATGAACGCCTTCACTGATATCCAAAGCATGAAGGACACACCAG...,TTCCCTGGTGTGTCCTTCATGCTTTGGATATCAGTGAAGGCGTTCA...,0.48,3,0.48,0.48,True,CCAATGAACGCCTTCACTGATATCC,AAAGCATGAAGGACACACCAGGGAA,CCTTGGCACCCGAGAATTCCATTCCCTGGTGTGTCCTTCATGCTTT,/5Phos/GGATATCAGTGAAGGCGTTCATTGGCCCATATAAGAAA,H19
1,chr9,22050400,22050450,CDKN2B-AS1_8_55610,.,+,chr9:22050400-22050450(+),GAAGTACTTCTTTGGCTGCCAAGGAAGTCCCAGCTGAAAGGTAACC...,CTTTGGTTACCTTTCAGCTGGGACTTCCTTGGCAGCCAAAGAAGTA...,0.48,3,0.48,0.48,True,GAAGTACTTCTTTGGCTGCCAAGGA,AGTCCCAGCTGAAAGGTAACCAAAG,CCTTGGCACCCGAGAATTCCACTTTGGTTACCTTTCAGCTGGGACT,/5Phos/TCCTTGGCAGCCAAAGAAGTACTTCCCCATATAAGAAA,H19
2,chr9,22050830,22050880,CDKN2B-AS1_8_56040,.,+,chr9:22050830-22050880(+),CCTTCAAATAGTTCACCTCAGTGGGATCCAGTGTGTGACCTGCAAA...,GCACTTTGCAGGTCACACACTGGATCCCACTGAGGTGAACTATTTG...,0.50,3,0.52,0.48,True,CCTTCAAATAGTTCACCTCAGTGGG,ATCCAGTGTGTGACCTGCAAAGTGC,CCTTGGCACCCGAGAATTCCAGCACTTTGCAGGTCACACACTGGAT,/5Phos/CCCACTGAGGTGAACTATTTGAAGGCCCATATAAGAAA,H19
